# Fedora spec VCS and URL field analysis

This notebook is for analyzing the adoption and contents of the `VCS` and `URL` RPM specfile metadata fields in Fedora Packages for the purpose of investivating the existence/reliability of upstream URL data.

## Imports and helpers

In [ ]:

import re
import os
import csv
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from urllib.parse import urlsplit
import requests
import tarfile


# https://stackoverflow.com/questions/16694907/download-a-large-file-in-python-with-requests
def download_file(url):
    local_filename = url.split('/')[-1]
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(local_filename, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192): 
                # If you have chunk encoded response uncomment if
                # and set chunk_size parameter to None.
                #if chunk: 
                f.write(chunk)
    return local_filename


def stream_spec_contents(tar_path):
    """Yields the raw string contents of spec files directly from the tarball."""
    print("Opening tarball and streaming files to RAM...")
    with tarfile.open(tar_path, "r:xz") as tar:
        for member in tar:
            if member.isfile() and member.name.endswith(".spec"):
                file_bytes = tar.extractfile(member)
                if file_bytes is not None:
                    # 'replace' handles any old non-UTF-8 characters safely without crashing
                    yield file_bytes.read().decode("utf-8", errors="replace")
spec_tag_map = {
    "VCS": "vcs",
    "URL": "url",
    "NAME": "name",
    "VERSION": "version",
    "RELEASE": "release",
    "LICENSE": "license",
    "SUMMARY": "summary"
}

def parse_single_spec(spec_content):
    spec_data = {

    }
    
    for line in spec_content.splitlines():
        if line.strip() == "":
            continue
        if ":" not in line:
            continue
        tag = line.split(":", 1)[0]
        tag_data = line[len(tag) + 1:].strip()
        if not spec_tag_map.get(tag.upper()):
            continue
        spec_data[spec_tag_map.get(tag.upper())] = tag_data
    return spec_data

CSV_PATH = Path("specfile_data.csv")

MACRO_RE = re.compile(r"%\{[^}]+\}")

## Data download and prep

In [ ]:

# 1. Handle Archive Download
archive_path = Path("rpm-specs-latest.tar.xz")
if not archive_path.exists():
    print("Downloading archive...")
    archive_path = download_file("https://src.fedoraproject.org/lookaside/rpm-specs-latest.tar.xz")

# 2. Parallel Processing Pipeline
print("Processing Specfiles...")
spec_data = []

for f in stream_spec_contents(archive_path):
    spec_data.append(parse_single_spec(f))

print(f"Successfully parsed {len(spec_data)} spec files total.")

# 3. Save directly to CSV
print("Saving intermediate CSV...")
if spec_data:
    with open(CSV_PATH, "w", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=spec_tag_map.values())
        writer.writeheader()
        writer.writerows(spec_data)


df = pd.read_csv(CSV_PATH)
print("Done")

In [ ]:

def categorize_vcs(value: object) -> str:
    if pd.isna(value):
        return "Not populated"

    text = str(value).strip()
    if not text:
        return "Not populated"
    if MACRO_RE.search(text):
        return "Populated with macros"
    parts = value.split(":", 1)
    if parts[0] in ["git", "hg", "svn", "cvs"]:
        return "Valid"
    print(value)
    return "Populated normally"

# Optional: only count packages whose specfile was fetched successfully.
# df = df[df["fetch_error"].fillna("").eq("")]

df["vcs_category"] = df["vcs"].map(categorize_vcs)

# df["vcs_category_detailed"].value_counts()


category_order = [
    "Dead Package",
    "Spec fetch failed",
    "Not populated",
    "Populated with macros",
    "Populated normally",
    "Valid"
]

counts = (
    df["vcs_category"]
    .value_counts()
    .reindex(category_order, fill_value=0)
)

summary = pd.DataFrame(
    {
        "category": counts.index,
        "count": counts.values,
        "percent": (counts.values / len(df) * 100).round(1),
    }
)

print(f"Rows: {len(df):,}")
# if "fetch_error" in df.columns:
#     failed = df["fetch_error"].fillna("").ne("").sum()
#     print(f"Spec fetch failures: {failed:,}")

summary

In [ ]:
colors = {
    "Dead Package": "#333",
    "Spec fetch failed": "#555",
    "Not populated": "#d62728",
    "Populated with macros": "#ff7f0e",
    "Populated normally": "#2ca02c",
    "Valid": "#00FF00"

}

fig, ax = plt.subplots(figsize=(8, 6))

y_pos = range(len(counts))
bars = ax.barh(
    y_pos,
    counts.values,
    color=[colors[label] for label in counts.index],
)

ax.set_yticks(y_pos)
ax.set_yticklabels(counts.index)
ax.invert_yaxis()
ax.set_xlabel("Count")
ax.bar_label(
    bars,
    labels=[
        f"{count:,} ({pct:.1f}%)"
        for count, pct in zip(counts.values, counts.values / len(df) * 100)
    ],
    padding=4,
)

ax.set_title("Fedora specfile VCS field adoption categories", fontsize=14, pad=16)
plt.tight_layout()
plt.show()

summary

## URL field adoption

In [ ]:

def categorize_url(value: object) -> str:
    if pd.isna(value):
        return "Not populated"

    text = str(value).strip()
    if not text:
        return "Not populated"
    if MACRO_RE.search(text):
        return "Populated with macros"
    return "Populated normally"

# Optional: only count packages whose specfile was fetched successfully.
# df = df[df["fetch_error"].fillna("").eq("")]

df["url_category"] = df["url"].map(categorize_url)



category_order = [
    "Dead Package",
    "Fetch Error",
    "Not populated",
    "Populated with macros",
    "Populated normally",
]

counts = (
    df["url_category"]
    .value_counts()
    .reindex(category_order, fill_value=0)
)

summary = pd.DataFrame(
    {
        "category": counts.index,
        "count": counts.values,
        "percent": (counts.values / len(df) * 100).round(1),
    }
)

print(f"Rows: {len(df):,}")
# if "fetch_error" in df.columns:
#     failed = df["fetch_error"].fillna("").ne("").sum()
#     print(f"Spec fetch failures: {failed:,}")

summary

In [ ]:
colors = {
    "Dead Package": "#333",
    "Fetch Error": "#555",
    "Not populated": "#d62728",
    "Populated with macros": "#ff7f0e",
    "Populated normally": "#2ca02c",
}

fig, ax = plt.subplots(figsize=(8, 6))

y_pos = range(len(counts))
bars = ax.barh(
    y_pos,
    counts.values,
    color=[colors[label] for label in counts.index],
)

ax.set_yticks(y_pos)
ax.set_yticklabels(counts.index)
ax.invert_yaxis()
ax.set_xlabel("Count")
ax.bar_label(
    bars,
    labels=[
        f"{count:,} ({pct:.1f}%)"
        for count, pct in zip(counts.values, counts.values / len(df) * 100)
    ],
    padding=4,
)

ax.set_title("Fedora specfile URL field adoption", fontsize=14, pad=16)
plt.tight_layout()
plt.show()

summary

## URLs by type

In [ ]:
 
def decide_url_type(value: object) -> str:
    if pd.isna(value):
        return "Empty"

    url = str(value).strip()
    if not url:
        return "Empty"

    urlparts = urlsplit(url)

    if urlparts.hostname is None:

        if MACRO_RE.search(url) or url.startswith("%"):
            if "forgeurl" in url or "git" in url:
                return "Known Forge"
            elif "gourl" in url:
                return "Questionable Link"
            elif "cran_url" in url or "bioc_url" in url or "ansible_collection_url" in url:
                return "Known Package Repo"
            # print(url)
            return "Populated with macros"
        # print(url)
        return "No Hostname"

    # known forges (likely to be the actual upstream url, pending further poking)
    forges = {
        "codeberg.org": "Codeberg",
        "gitlab.com": "Gitlab",
        "pagure.io": "Pagure",
        "gitlab.freedesktop.org": "Freedesktop gitlab",
        "gitlab.winehq.org": "wine gitlab",
        "bitbucket.org": "bitbucket",
        "sr.ht": "Sourcehut",
        "code.google.com": "Google Code",
        "github.com": "Github",
        "sourceforge.net": "Sourceforge",
        "sourceforge.jp": "Sourceforge",
        "invent.kde.org": "KDE invent gitlab",
        "gitlab.gnome.org": "GNOME gitlab",
        "git.gnome.org": "GNOME gitlab",
        "savannah.nongnu.org": "GNU",
        "savannah.gnu.org": "GNU",
        "googlesource.com": "google forge",
        "code.videolan.org": "videolan gitlab",
        "forge.fedoraproject.org": "fedora forgejo",
        "src.fedoraproject.org": "fedora dist-git",
        "forge.moderndesktop.dev": "someones forgejo",
        "sourceware.org": "Sourceware",
        "salsa.debian.org": "debian gitlab",
        "source.puri.sm": "purism gitlab",

    }

    if forges.get(urlparts.hostname):
        return "Known Forge"
    
    # Check for known forge end matches   
    for k in forges.keys():
        if urlparts.hostname.endswith(k):
            return "Known Forge"
        if urlparts.hostname.startswith("gitea.") or urlparts.hostname.startswith("gitlab."):
            return "Known Forge"

    



    # Storefronts where the upstream URL is maybe parseable
    storefronts = {
        "extensions.gnome.org": "GNOME Extensions",
        "apps.gnome.org": "Gnome apps",
        "extensions.openoffice.org": "openoffice extensions",
        "addons.mozilla.org": "firefox addons",
    }

    if storefronts.get(urlparts.hostname):
        return "Storefront"

    if "kde.org/applications" in url or "gnu.org/software" in url:
    #urlparts.hostname.endswith("kde.org") and urlparts.path.startswith("/applications"):
        return "Storefront"
    



    # package repos where the upstream URL is likely parseable
    packagerepos = {
        "crates.io": "Rust/cargo",
        "pypi.org": "Pypi",
        "pypi.python.org": "Pypi",
        "pypi.io": "Pypi",
        "hackage.haskell.org": "haskell",
        "rubygems.org": "Ruby",
        "packages.debian.org": "Debian pkg",
        "maven.apache.org": "Maven",
        "cpan.org": "CPAN",
        "ctan.org": "CTAN",
    }
    if packagerepos.get(urlparts.hostname):
        return "Known Package Repo"
    
    # Check for known forge end matches   
    for k in packagerepos.keys():
        if urlparts.hostname.endswith(k):
            return "Known Package Repo"

    



    # docs and project pages. very unlikely to be useful without substantial effort. AI may be good for finding upstreams in these
    docs_project_pages = {
        "github.io": "Github.io",
        "sourceforge.io": "Sourceforge",
        "readthedocs.io": "ReadTheDocs",
        "readthedocs.org": "ReadTheDocs",
        "apps.kde.org": "kde project pages",
        "wiki.gnome.org": "gnome wiki",
        "community.kde.org": "KDE Wiki",
        "userbase.kde.org": "KDE userbase wiki",
        "techbase.kde.org": "kde techbase wiki",
        "wiki.services.openoffice.org": "",
        "en.opensuse.org": "opensuse wiki",
        "commons.apache.org": "apache wiki",
        "software.sil.org": "homepage",

    }
    if docs_project_pages.get(urlparts.hostname):
        return "Known Docs or Project Page"
    
    # Check for known forge end matches   
    for k in docs_project_pages.keys():
        if urlparts.hostname.endswith(k):
            return "Known Docs or Project Page"

    if "fedoraproject.org/wiki" in url:
        return "Known Docs or Project Page"
    
    if urlparts.hostname.startswith("docs."):
        return "Known Docs or Project Page"





    # Misc

    misc = {
        # "kde.org": "KDE",
        # "gnome.org": "GNOME",
        "php.net": "PHP",
        "xfce.org": "XFCE",
        "launchpad.net": "launchpad.net",
        "freedesktop.org": "Freedesktop",
    }

    if misc.get(urlparts.hostname):
        return "Known Misc"
    
    # Check for known forge end matches   
    for k in misc.keys():
        if urlparts.hostname.endswith(k):
            return "Known Misc"


    deadlinks = {
        "projects.gnome.org": "gnome wiki old links",
        "cgit.kde.org": "old kde cgit", #site doesnt exist
        "extensions.services.openoffice.org": "gone",
        "forge.ocamlcore.org": "gone",
        "gitorious.org": "slow to respond",

    }
    if deadlinks.get(urlparts.hostname):
        return "Dead Link"
    
    if urlparts.hostname == "www.gnome.org" and (urlparts.path.startswith("/projects") or urlparts.path.startswith("/fonts")):
        return "Dead Link"
    elif urlparts.hostname == "www.gnome.org":
        return "Known Docs or Project Page"

    if urlparts.hostname == "fedorahosted.org":
        return "Old Link"


    if urlparts.scheme == "ftp":
        return "FTP"
    
    if urlparts.scheme == "git":
        return "git:custom"

    if urlparts.path == "" or urlparts.path == "/":
        return "Homepage"


    if "kde" in urlparts.hostname or "gnome" in urlparts.hostname:
        # print(url)
        return "Misc"

    if urlparts.hostname.startswith("git."):
        return "git:custom"

    if urlparts.hostname.startswith("wiki."):
        return "wiki"
    
    if urlparts.hostname.startswith("ftp."):
        return "FTP"
    
    # print(url)
    return "Uncategorized"


category_col = "url_type"

df[category_col] = df["url"].map(decide_url_type)



def get_hostname(d):
    if pd.isna(d):
        return ""
    try:
        return urlsplit(d).hostname
    except Exception as e:
        print(d)
        print(e.getmessage())
        return ""


def get_domain(d):
    host = get_hostname(d)
    if not host or host == "":
        return ""
    
    hostspl = host.split(".")
    if len(hostspl) > 2:
        return f"{hostspl[-2]}.{hostspl[-1]}"
    else:
        return host 

df["hostname"] = df["url"].map(get_domain)

undetected_hostnames = df.loc[df[category_col] == "Uncategorized", 'hostname']


# Any categorical column works; adjust if yours is named differently.
counts = df[category_col].value_counts(dropna=False)


# Sort by count (largest first).
counts = counts.sort_values(ascending=False)

# Dynamic colors: pick one color per category from a matplotlib colormap.
cmap = plt.get_cmap("tab10")  # or "Set2", "viridis", etc.
colors = [cmap(i / max(len(counts) - 1, 1)) for i in range(len(counts))]

fig, ax = plt.subplots(figsize=(10, max(6, len(counts) * 0.35)))

y_pos = range(len(counts))
bars = ax.barh(y_pos, counts.values, color=colors)

ax.set_yticks(y_pos)
ax.set_yticklabels(counts.index)
ax.invert_yaxis()
ax.set_xlabel("Count")
ax.bar_label(
    bars,
    labels=[
        f"{count:,} ({pct:.1f}%)"
        for count, pct in zip(counts.values, counts.values / counts.sum() * 100)
    ],
    padding=4,
    fontsize=9,
)

ax.set_title(f"Distribution of {category_col}", fontsize=14, pad=16)
plt.tight_layout()
plt.show()

# Optional: summary table with the same dynamic ordering
summary = (
    counts.rename("count")
    .to_frame()
    .assign(percent=lambda s: (s["count"] / s["count"].sum() * 100).round(1))
)
summary


# This section is url spelunking and EDA.

#  per-row: keep hostname if seen >1 time, else bucket singletons
hostname_counts = undetected_hostnames.value_counts(dropna=False)

hostname_group = undetected_hostnames.map(
    lambda h: h if hostname_counts.get(h, 0) > 11 else f"Uncategorized (used {hostname_counts.get(h, 0)} times)" #("Uncategorized (1)" if hostname_counts.get(h, 0) == 1 else "multi-occurrence domains")
)
# # counts for pie chart
grouped_counts = hostname_group.value_counts().sort_values(ascending=False)
grouped_counts
# df.loc[df["hostname"] == ""]
